# MuJoCo URDF Import
Onshape can export assemblies in the [Universal Robotic Description Format (URDF)](https://wiki.ros.org/urdf). MuJoCo only has limited support for working with these files directly [(docs section)](https://mujoco.readthedocs.io/en/stable/modeling.html#curdf), so instead parse some key information from it to create a MuJoCo-native MJCF XML file.

## Config

In [2]:
import sys
sys.modules["tensorboard"] = None
sys.modules["tensorflow"] = None

from pathlib import Path

import mujoco as mj

ONSHAPE_ASSEMBLY = "assembly_simplified"
OUTPUT = "robot.xml"
# ONSHAPE_ASSEMBLY = "assembly_static"
# OUTPUT = "robot-static.xml"
# ONSHAPE_ASSEMBLY = "assembly_bala2ish"
# OUTPUT = "bala2ish.xml"

mass_override = {
	"electronics": 0.032,
	# "electronics": 0.02,
	"wheel": 0.020,
	"wheel_1": 0.020,
	"sensor_imu": 0,
	"topweight": 0.215,
}

cwd = Path(".")
path_assembly = cwd / ONSHAPE_ASSEMBLY
path_urdf = path_assembly / "urdf" / (ONSHAPE_ASSEMBLY + ".urdf")
path_meshes = path_assembly / "meshes"

In [3]:
def mapl(items, func):
	return list(map(func, items))


def parse_vec(string):
	return mapl(string.split(" "), lambda x: float(x))

## Parsing

In [4]:
import xml.etree.ElementTree as ET


xml = ET.parse(path_urdf)

robot = xml.getroot()

links = {}

print("Links (bodies):")
for link in robot.findall("./link"):
	name = link.get("name")
	print(f"  {name}")
	links[name] = {"xml": link, "children": []}

print()
print("Joints:")
for joint in robot.findall("./joint"):
	joint_name = joint.get("name")
	joint_type = joint.get("type")
	if not (joint_type == "fixed" or joint_type == "continuous"):
		print(f"  '{joint_name}' unsupported mate type: {joint_type}")
		continue
	parent = joint.find("./parent").get("link")
	child = joint.find("./child").get("link")
	print(f"  '{joint_name}' {joint_type} {child} to {parent}")
	p = links.get(parent)
	mate = {
		"joint_name": joint_name,
		"joint_type": joint_type,
		"joint_axis": parse_vec(joint.find("./axis").get("xyz")),
		"origin": parse_vec(joint.find("./origin").get("xyz")),
		"rpy": parse_vec(joint.find("./origin").get("rpy")),
		"child": child,
	}
	p["children"].append(mate)


Links (bodies):
  electronics
  lipos
  mark_wheel
  mark_wheel_1
  motor_base
  root
  sensor_imu
  sketch_wheelbase__1_
  wheel
  wheel_1

Joints:
  'fastened_1' fixed motor_base to root
  'fastened_2' fixed sketch_wheelbase__1_ to root
  'fastened_2_1' fixed lipos to motor_base
  'fastened_3' fixed electronics to lipos
  'fastened_4' fixed sensor_imu to electronics
  'fastened_5' fixed mark_wheel_1 to wheel_1
  'fastened_6' fixed mark_wheel to wheel
  'wheel_left' continuous wheel_1 to sketch_wheelbase__1_
  'wheel_right' continuous wheel to sketch_wheelbase__1_


In [5]:
import math
from scipy.spatial.transform import Rotation

spec = mj.MjSpec.from_file("template.xml")

spec.compiler.meshdir = str(path_meshes)


def convert(mate, parent=spec.worldbody):
	name = mate.get("child")
	link = links.get(name)
	print(name, mate, link)

	has_rot = False
	for x in mate["rpy"]:
		if abs(x) > 1e-3:
			has_rot = True
			break

	euler=[0, 0, 0]
	if has_rot:
		euler = [ round(x/math.pi)*180 for x in mate["rpy"]]
		print("  joint rotation", euler)
	rot = Rotation.from_euler("xyz", euler, degrees=True)
	quat = rot.as_quat(scalar_first=True)

	inertial = link["xml"].find("./inertial")
	body: mj.MjsBody = parent.add_body(
		name=name,
		pos = mate["origin"],
		quat=quat
	)
	mesh = link["xml"].find("./visual/geometry/mesh")
	color = link["xml"].find("./visual/material/color")
	if color is not None:
		print("  ", "color", color)
		color = parse_vec(color.get("rgba"))

	if mesh is not None:
		if inertial is None:
			print(f"error: mesh {name} has no inertial info")
			return

		mass = float(inertial.find("mass").get("value"))

		mesh_file = Path(mesh.get("filename").replace("package://", "")).relative_to(
			path_meshes
		)
		mesh_name = name
		spec.add_mesh(
			name=mesh_name, file=str(mesh_file), scale=parse_vec(mesh.get("scale"))
		)
		spec.assets[mesh_name] = str(mesh_file)
		geom_name = f"geom_{name}"
		pos = parse_vec(link["xml"].find("./visual/origin").get("xyz"))
		opts = dict(
				name=geom_name,
				rgba=color,
				type=mj.mjtGeom.mjGEOM_MESH,
				meshname=mesh_name,
				mass=mass_override.get(name, mass),
				pos=pos,
				# euler=euler
			)

		if name.startswith("sensor_") or name.startswith("mark_"):
			opts["mass"] = 0

		body.add_geom(
			**opts
		)

	else:
		print(f"  {name} has no geometry")


	joint_name = mate["joint_name"]
	match mate["joint_type"]:
		case "freejoint":
			body.add_freejoint(name=joint_name)
			pass
		case "fixed":
			pass  # mujoco default
		case "continuous":
			body.add_joint(
				name=f"joint_{joint_name}",
				type=mj.mjtJoint.mjJNT_HINGE,
				axis=mate["joint_axis"],
				#  armature: motor's rotational inertia reflected through the gearbox
				#            Estimated rotational inertia of N20 motor: 5e-8 Kg-m^2.
				#            With a 1:G gearbox, we multiply the inertia by G^2 to get
				#            the reflected inertia. So, 5e-8 * 30^2 = 4.5e-5 Kg-m^2.
				#            This is ~10x the wheel's inertia, so this dominates.
				# 30:1 + 2:1 gearing
				armature=18e-5,
			)

			body.add_joint(
				name=f"slop_joint_{joint_name}",
				type=mj.mjtJoint.mjJNT_HINGE,
				axis=mate["joint_axis"],
				limited=True,
				range=[-4.584, 4.584], # degrees
			)

			act = spec.add_actuator(name=f"act_{joint_name}")
			act.damping = [0, 0, 0] # messes with dcmotor..

			voltage = 8 # V
			no_load_speed = 34 # rad/s
			K = voltage/no_load_speed # motor constant
			resistance=11.5 # ohms

			act.set_to_dcmotor(
				motorconst=[K, K],
				resistance=resistance,
				inductance=[
					# 0.0026,  # 2.6mH based on https://www.klikrobotics.com/PWMTutorial2.html
					0.00248, # 2.48mH @ 500Hz measured for 500RPM encoder motor
					0,
				],
			)
			t_max = (voltage/resistance)*act.gainprm[1]
			act.forcerange = [-t_max, t_max]
			act.ctrlrange = [-8, 8]

			act.target = f"joint_{joint_name}"
			act.trntype = mj.mjtTrn.mjTRN_JOINT

			spec.add_sensor(
				name=f"jointvel_{joint_name}",
				type=mj.mjtSensor.mjSENS_JOINTVEL,
				objtype=mj.mjtObj.mjOBJ_JOINT,
				objname=f"joint_{joint_name}",
			)

	if name.startswith("sensor_"):
		body.add_site(name=f"site_{name}", pos=[0, 0, 0], size=[0.001, 0.001, 0.001])
		sensor_type = name.replace("sensor_", "")
		if sensor_type == "imu":
			spec.add_sensor(
				name="imu_quat",
				type=mj.mjtSensor.mjSENS_FRAMEQUAT,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)
			spec.add_sensor(
				name="imu_accel",
				type=mj.mjtSensor.mjSENS_ACCELEROMETER,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)
			spec.add_sensor(
				name="imu_gyro",
				type=mj.mjtSensor.mjSENS_GYRO,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)

	for c in link["children"]:
		convert(c, body)


convert(
	{
		"joint_name": "_root",
		"joint_type": "freejoint",
		"origin": [0, 0, 0.01],
		"rpy": [0, 0, 0],
		"child": "root",
	}
)

# privileged velocity info
spec.add_sensor(
	name="priv_vel",
	type=mj.mjtSensor.mjSENS_VELOCIMETER,
	objtype=mj.mjtObj.mjOBJ_SITE,
	objname="site_sensor_imu",
)

print()
print(spec.to_xml())

spec.encode(OUTPUT)


root {'joint_name': '_root', 'joint_type': 'freejoint', 'origin': [0, 0, 0.01], 'rpy': [0, 0, 0], 'child': 'root'} {'xml': <Element 'link' at 0x7d5e6b815ee0>, 'children': [{'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'rpy': [0.0, -0.0, 0.0], 'child': 'motor_base'}, {'joint_name': 'fastened_2', 'joint_type': 'fixed', 'joint_axis': [0.0, 0.0, 1.0], 'origin': [0.0, 0.0, 0.0], 'rpy': [0.0, -0.0, 0.0], 'child': 'sketch_wheelbase__1_'}]}
  root has no geometry
motor_base {'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'rpy': [0.0, -0.0, 0.0], 'child': 'motor_base'} {'xml': <Element 'link' at 0x7d5e6b8164d0>, 'children': [{'joint_name': 'fastened_2_1', 'joint_type': 'fixed', 'joint_axis': [-1.13185e-16, -2.46519e-32, -1.0], 'origin': [-1.92593e-34, -3.41524e-18, 0.0481542], 'rpy': [1.35525e-16, 6.93335e-33, 0.0], 'child': 'lipos'}]}
   color <Element 'color' at 

4769

In [6]:
import mujoco as mj

spec = mj.MjSpec.from_file(OUTPUT)
model = spec.compile()
data = mj.MjData(model)

mj.mj_forward(model, data)

chassis = data.body("motor_base")

com = chassis.subtree_com * 1000

axlepos = (data.body("wheel").xpos + data.body("wheel_1").xpos) / 2

print(OUTPUT)
print(f"  mass {model.body('root').subtreemass[0] * 1000:.4f} gram")
print(f"  center of mass [ {com[0]:.4f} {com[1]:.4f} {com[2]:.4f} ] mm")
print(
	f"  axle position {axlepos[2] * 1000:.4f} mm [ {axlepos[0]:.4f} {axlepos[1]:.4f} {axlepos[2]:.4f} ]"
)
print(
	f"  pendulum arm {(chassis.subtree_com[2] - data.body('wheel').xpos[2]) * 1000:.4f} mm"
)

site = data.site("site_sensor_imu")
relpos = (site.xpos - axlepos) * 1000
print(
	f"  IMU pos relative to axle [ {relpos[0]:.4f} {relpos[1]:.4f} {relpos[2]:.4f} ] mm"
)

print()

robot.xml
  mass 156.9138 gram
  center of mass [ 0.0535 0.0445 62.8188 ] mm
  axle position 40.0000 mm [ 0.0000 0.0000 0.0400 ]
  pendulum arm 22.8188 mm
  IMU pos relative to axle [ 12.0000 5.0000 51.2042 ] mm



# Launch MuJoCo

In [7]:
import mujoco as mj
import mujoco.viewer

spec = mj.MjSpec.from_file(OUTPUT)
model = spec.compile()
data = mj.MjData(model)

mujoco.viewer.launch(model, data)


(python:12644): dbind-WARNING **: 19:12:45.786: AT-SPI: Error retrieving accessibility bus address: org.freedesktop.DBus.Error.ServiceUnknown: The name is not activatable

(python:12644): Gtk-WARNING **: 19:12:45.788: Theme parsing error: gtk.css:1048:16: 'border-spacing' is not a valid property name

(python:12644): Gtk-WARNING **: 19:12:45.810: Theme parsing error: gtk.css:1048:16: 'border-spacing' is not a valid property name
/nix/store/zlm2a41vig3441m9b27y33ib9zyg2icc-python3-3.13.15-env/lib/python3.13/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


In [24]:
import mujoco as mj
import mujoco.viewer
import numpy as np

spec = mj.MjSpec.from_file(OUTPUT)
model = spec.compile()
data = mj.MjData(model)

mj.mj_resetData(model, data)

# Impart an initial angular velocity around the x axis so the agent learns to recover
data.qvel[3] += np.random.uniform(-2.5, 2.5)
# and y axis
data.qvel[4] += np.random.uniform(-2.5, 2.5)

mj.mj_forward(model, data)


mujoco.viewer.launch(model, data)

KeyboardInterrupt: 

In [113]:
import time
import atexit

import mujoco as mj
import mujoco.viewer as mjv

from types import SimpleNamespace
from scipy.spatial.transform import Rotation
import numpy as np
from oscpy.client import OSCClient

spec = mj.MjSpec.from_file(OUTPUT)
model = spec.compile()
data = mj.MjData(model)

try:
	viewer.close()
except:
	pass

step = 0

a1m = model.actuator("act_wheel_left")
a2m = model.actuator("act_wheel_right")

a1 = data.actuator("act_wheel_left")
a2 = data.actuator("act_wheel_right")
jv1 = data.sensor("jointvel_wheel_left")
jv2 = data.sensor("jointvel_wheel_right")
robot = data.body("root")

d = SimpleNamespace(
			robot=data.body("root"),  # to get position
			act1=data.actuator("act_wheel_left"),
			act2=data.actuator("act_wheel_right"),
			imu_gyro=data.sensor("imu_gyro"),
			imu_quat=data.sensor("imu_quat"),
			priv_vel=data.sensor("priv_vel"),
			input="voltage"
		)

# a1m.gear[0] = 0.8
# print(a1m.gear, a2m.gear)
# return

osc = OSCClient("127.0.0.1", 1235)
osc2 = OSCClient("192.168.1.143", 1212)

mj.mj_forward(model, data)

vels = []

viewer = mjv.launch_passive(model, data)

@atexit.register
def end():
	osc.send_message(b"/rec", [0])
	osc2.send_message(b"/act", [0.0, 0.0])

MAX_RPS = 37

def osc_update():
	print(f"m1: act {a1.ctrl[0]/a1m.ctrlrange[1]}, vel {jv1.data[0]/MAX_RPS}, m2: act {a2.ctrl[0]/a2m.ctrlrange[1]}, vel {jv2.data[0]/MAX_RPS}")
	osc.send_message(b'/motor_sim', [float(jv1.data[0]/MAX_RPS), float(jv2.data[0]/MAX_RPS), float(a1.ctrl[0]/a1m.ctrlrange[1]), float(a2.ctrl[0]/a2m.ctrlrange[1])])

def drive(m1, m2):
	a1.ctrl = m1 * a1m.ctrlrange[1]
	a2.ctrl = m2 * a2m.ctrlrange[1]
	osc2.send_message(b'/act', [float(m1), float(m2)])

osc.send_message(b"/rec", [1])

while True:
	# if step == 0:
	# 	drive(1, 1)
	# if step == 500:
	# 	drive(0, 0)
	# if step == 1000:
	# 	drive(0, 0)
	# if step == 1200:
	# 	drive(0.2, 0.2)
	# if step == 1400:
	# 	drive(0.4, 0.4)
	# if step == 1600:
	# 	drive(0.6, 0.6)
	# if step == 1800:
	# 	drive(0.8, 0.8)
	# if step >= 2000:
	# 	if step % 50 == 0:
	# 		drive(1.0, 1.0)
	# 		if step % 100 == 0:
	# 			drive(-1, -1)
	# if step == 2500:
	# 	osc.send_message(b"/rec", [0])
	# 	for i in range(10):
	# 		drive(0, 0)
	# 		time.sleep(0.5)
	# 	break

	# osc_update()
	drive(0.15, 0.15)

	step += 1
	step_start = time.time()
	mj.mj_step(model, data)
	viewer.sync()

	gyro = d.imu_gyro.data
	quat = d.imu_quat.data

	# r = Rotation.from_quat(quat, scalar_first=True)
	# r.inv()



	# eul = r.as_euler("xyz", degrees=False, suppress_warnings=True)
	# if step % 30 == 0:
	# 	print(step, "vel", jv1.data[0], jv2.data[0])
		# print(step, "eul", eul)
		# print(" " * (len(f"{step}")), "vel", d.priv_vel.data)


	# if step == 400:
	# 	pos = robot.xpos[1]
	# if step > 400:
	# 	vels.append(jv1.data[0])
	# if step > 800:
	# 	print("moved", (robot.xpos[1] - pos)/2, "m/s")
	# 	print("avg vel steps 400-800", np.mean(vels)/(2*np.pi), "rotations per second")
	# 	viewer.close()
	# 	time.sleep(1)
	# 	break

	# print(step, j1.qvel, j1.qacc)

	sleep = model.opt.timestep - (time.time() - step_start)
	# sleep = 0.001 - (time.time() - step_start)
	if sleep > 0:
		time.sleep(sleep)


/nix/store/zlm2a41vig3441m9b27y33ib9zyg2icc-python3-3.13.15-env/lib/python3.13/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


KeyboardInterrupt: 

In [44]:
viewer.close()